# 📦 Aula 10 — Hipergeométrica e binomial no dia a dia de um time de dados

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **time de dados do setor de Qualidade de uma loja online**. O setor precisa tomar três decisões, e cada uma é respondida com uma ferramenta da Aula 10: a **distribuição hipergeométrica** (sorteio **sem reposição**) e a **distribuição binomial** (ensaios **independentes** com a mesma probabilidade).

> ⚠️ Os números (pedidos, erros, taxa de clique) são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`math.comb`**: calcula combinações $\binom{n}{k}$, que aparecem nas fórmulas das duas distribuições;
- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**.

Também escrevemos, uma única vez, as **duas fórmulas da aula** como funções do Python. Uma função é só uma "receita" com nome: depois é só chamar `hiper(...)` ou `binom(...)` com os números do problema.

- $P(X=k) = \dfrac{\binom{r}{k}\binom{N-r}{n-k}}{\binom{N}{n}}$ — **hipergeométrica** ($N$ = população, $r$ = elementos com a característica, $n$ = amostra);
- $P(X=k) = \binom{n}{k}\, p^k (1-p)^{n-k}$ — **binomial** ($n$ = ensaios, $p$ = probabilidade de sucesso).

Por fim, criamos o **sorteador** (`rng`) com a **semente 42**, que faz as simulações saírem sempre iguais.

**Lembrete:** **empírico** = o que sai da simulação; **teórico** = o que a fórmula da aula diz.

In [ ]:
from math import comb                 # combinações C(n, k)
import numpy as np                    # calculadora de números e sorteios
import matplotlib.pyplot as plt       # gráficos


def hiper(k, N, r, n):
    # P(X = k) na hipergeométrica: população N, r com a característica, amostra de n sem reposição
    return comb(r, k) * comb(N - r, n - k) / comb(N, n)


def binom(k, n, p):
    # P(X = k) na binomial: n ensaios independentes, probabilidade de sucesso p
    return comb(n, k) * p**k * (1 - p)**(n - k)


rng = np.random.default_rng(42)       # sorteador, semente 42
print("Tudo pronto!")

## 1. A auditoria de pedidos

### A situação
Hoje a loja recebeu **50 pedidos**. Pelo histórico, o setor sabe que **6 deles têm erro de cadastro** (endereço incompleto, CPF inválido...), mas **não sabe quais**. O auditor só tem tempo para conferir **10 pedidos**, sorteados ao acaso — e, claro, não confere o mesmo pedido duas vezes.

### A pergunta
Seja $X$ = número de pedidos com erro entre os 10 auditados.
1. Qual a chance de o auditor encontrar **pelo menos 1** pedido com erro?
2. Quantos pedidos com erro ele encontra, **em média**?

### A ferramenta da aula
O sorteio é **sem reposição**, de uma população **finita** dividida em dois grupos (com erro / sem erro). Então $X$ é **hipergeométrica** com $N = 50$, $r = 6$ e $n = 10$.

### A resposta pela conta
- Pelo complementar: $P(X \ge 1) = 1 - P(X = 0) = 1 - \dfrac{\binom{6}{0}\binom{44}{10}}{\binom{50}{10}} \approx 1 - 0{,}2415 = 0{,}7585$.
- Esperança: $E(X) = n \cdot \dfrac{r}{N} = 10 \cdot \dfrac{6}{50} = 1{,}2$.

### O que o código faz
1. calcula as duas respostas com a função `hiper`;
2. **simula 100.000 dias de auditoria**: em cada dia, o sorteador escolhe 10 dos 50 pedidos, sem reposição (`rng.hypergeometric` faz exatamente isso), e conta quantos têm erro;
3. compara a fração de dias com pelo menos 1 erro, e a média de erros, com a conta.

In [ ]:
N, r, n = 50, 6, 10                                    # 50 pedidos, 6 com erro, 10 auditados

p_pelo_menos_1 = 1 - hiper(0, N, r, n)                 # complementar de "nenhum erro"
esperanca = n * r / N

# simulação: 100.000 dias; em cada um, 10 pedidos sorteados sem reposição
erros_encontrados = rng.hypergeometric(r, N - r, n, size=100_000)

print(f"P(X >= 1)  teórico: {p_pelo_menos_1:.4f}   empírico: {(erros_encontrados >= 1).mean():.4f}")
print(f"E(X)       teórico: {esperanca:.2f}     empírico: {erros_encontrados.mean():.2f}")

**Como ler:** a conta e a simulação concordam: há cerca de **76%** de chance de o auditor encontrar pelo menos um pedido com erro, e em média ele encontra **1,2** erros. Dito de outro jeito: em cerca de **1 a cada 4 dias**, a auditoria não encontra nenhum dos 6 erros, mesmo eles existindo.

Para ver a distribuição inteira, o código abaixo calcula $P(X=k)$ para cada número possível de erros encontrados (de 0 a 6) e desenha um **gráfico de barras**: a altura de cada barra é a probabilidade daquele valor.

In [ ]:
valores = range(0, 7)                                  # X pode ir de 0 a 6
probs = [hiper(k, N, r, n) for k in valores]           # P(X = k) para cada k

for k, pk in zip(valores, probs):
    print(f"P(X = {k}) = {pk:.4f}")

plt.bar(valores, probs, color="steelblue", edgecolor="black")
plt.xlabel("Pedidos com erro encontrados entre os 10 auditados")
plt.ylabel("Probabilidade")
plt.title("Distribuição hipergeométrica: N = 50, r = 6, n = 10")
plt.show()

**Como ler:** o mais provável é encontrar **1 erro** (cerca de 41%), e a barra do **zero** (cerca de 24%) é a segunda maior. Encontrar 4 ou mais erros é raríssimo. Isso mostra o limite de uma auditoria por amostragem: ela quase nunca revela todos os problemas de uma vez.

## 2. A decisão do gerente: quantos pedidos auditar?

### A situação e a pergunta
Para o gerente do setor, 76% de chance de detectar algum erro é pouco. Ele quer **pelo menos 90%** de chance de flagrar pelo menos 1 pedido com erro por dia. **Quantos pedidos, no mínimo, precisam ser auditados?**

### A ferramenta da aula
A mesma hipergeométrica, agora com o tamanho da amostra $n$ variando: queremos o **menor** $n$ tal que
$$P(X \ge 1) = 1 - \frac{\binom{6}{0}\binom{44}{n}}{\binom{50}{n}} \ge 0{,}90 .$$

### O que o código faz
Testa todos os tamanhos de auditoria, de 1 a 50 pedidos: para cada um, calcula a chance de detectar pelo menos 1 erro. Depois:
1. encontra o primeiro $n$ em que a chance passa de 90%;
2. desenha a curva "pedidos auditados × chance de detectar", com uma linha na meta de 90%.

In [ ]:
tamanhos = range(1, N + 1)                                     # auditar 1, 2, ..., 50 pedidos
chance = [1 - hiper(0, N, r, m) for m in tamanhos]             # P(pelo menos 1 erro) para cada tamanho

n_minimo = next(m for m, c in zip(tamanhos, chance) if c >= 0.90)   # primeiro tamanho que bate a meta
print(f"Auditando {n_minimo - 1} pedidos: {chance[n_minimo - 2]:.1%} de chance")
print(f"Auditando {n_minimo} pedidos: {chance[n_minimo - 1]:.1%} de chance  <- menor tamanho que bate a meta")

plt.plot(tamanhos, chance, color="steelblue", marker="o", markersize=3)
plt.axhline(0.90, color="firebrick", linestyle="--", label="meta de 90%")
plt.axvline(n_minimo, color="gray", linestyle=":")
plt.xlabel("Número de pedidos auditados (n)")
plt.ylabel("Chance de detectar pelo menos 1 erro")
plt.title("Quanto mais pedidos auditados, maior a chance de detectar")
plt.legend()
plt.show()

**Como ler:** com 15 pedidos, a chance é de 89,8% — quase lá, mas abaixo da meta. Com **16 pedidos**, passa para **91,5%**. Essa é a recomendação do time de dados ao gerente: **auditar 16 dos 50 pedidos por dia**.

Repare no formato da curva: ela sobe rápido no começo e depois **achata**. Os primeiros pedidos auditados aumentam muito a chance de detecção; depois de uns 25, cada pedido a mais quase não muda nada. É o tipo de gráfico que ajuda a equilibrar **custo** (tempo do auditor) e **benefício** (detectar erros).

## 3. A campanha de e-mail do marketing

### A situação
O setor de marketing vai mandar um e-mail de promoção para **20 clientes**. Pelo histórico de campanhas, **15%** dos clientes que recebem o e-mail clicam no link, e cada cliente decide **independentemente** dos outros. A meta da campanha é de **pelo menos 5 cliques**.

### A pergunta
Seja $X$ = número de clientes que clicam.
1. Qual a chance de a campanha **bater a meta**?
2. Quantos cliques são esperados?

### A ferramenta da aula
São $n = 20$ ensaios de Bernoulli (cada cliente clica ou não), **independentes**, com a mesma probabilidade $p = 0{,}15$. Então $X$ é **binomial**: $X \sim B(20;\, 0{,}15)$.

### A resposta pela conta
- $P(X \ge 5) = 1 - \big[P(X=0) + P(X=1) + P(X=2) + P(X=3) + P(X=4)\big] \approx 0{,}1702$;
- $E(X) = n\,p = 20 \times 0{,}15 = 3$.

### O que o código faz
1. calcula $P(X \ge 5)$ somando as probabilidades de 0 a 4 com a função `binom` e usando o complementar;
2. **simula 100.000 campanhas** (`rng.binomial` sorteia quantos dos 20 clientes clicam em cada uma);
3. desenha a distribuição de $X$, com as barras que batem a meta em outra cor.

In [ ]:
n_clientes, p_clique = 20, 0.15

p_meta = 1 - sum(binom(k, n_clientes, p_clique) for k in range(0, 5))   # 1 - P(X <= 4)
cliques = rng.binomial(n_clientes, p_clique, size=100_000)               # 100.000 campanhas simuladas

print(f"P(X >= 5)  teórico: {p_meta:.4f}   empírico: {(cliques >= 5).mean():.4f}")
print(f"E(X)       teórico: {n_clientes * p_clique:.2f}     empírico: {cliques.mean():.2f}")

valores = range(0, 11)
probs = [binom(k, n_clientes, p_clique) for k in valores]
cores = ["firebrick" if k >= 5 else "steelblue" for k in valores]        # vermelho = bateu a meta
plt.bar(valores, probs, color=cores, edgecolor="black")
plt.xlabel("Número de cliques (de 20 e-mails)")
plt.ylabel("Probabilidade")
plt.title("Binomial B(20; 0,15) — em vermelho, os resultados que batem a meta")
plt.show()

**Como ler:** a chance de bater a meta é de só **17%**: em cerca de 5 de cada 6 campanhas, a meta não seria atingida. O número esperado de cliques é **3**, e no gráfico as barras mais altas estão em 2 e 3 cliques, bem longe da meta.

**Conclusão para o marketing:** com 20 e-mails e 15% de cliques, "pelo menos 5 cliques" é uma meta **pouco realista**. Ou a meta cai para algo perto de 3, ou a campanha precisa ir para mais clientes (aumentar $n$).

## 4. Quando a base é enorme, a binomial basta

### A situação e a pergunta
No fim do mês, a auditoria sorteia **10 pedidos entre 5.000**, dos quais **600 têm erro** — a mesma proporção de 12% da seção 1 ($6/50 = 600/5000 = 0{,}12$). O sorteio continua **sem reposição**, então o modelo certo é a hipergeométrica. Mas: **a binomial daria quase a mesma resposta?**

### A ferramenta da aula
Sem reposição, cada pedido retirado muda um pouco a proporção dos que sobram. Mas, numa base de 5.000, tirar 10 pedidos quase não muda nada: a chance de erro continua praticamente $0{,}12$ em cada sorteio, como se os sorteios fossem independentes. Por isso, comparamos a hipergeométrica com a binomial $B(10;\, 0{,}12)$.

### O que o código faz
1. calcula $P(X=k)$ pelas duas distribuições, lado a lado, para $k$ de 0 a 4;
2. repete a comparação para a base pequena da seção 1 ($N = 50$), para ver a diferença.

In [ ]:
print("Base de 5.000 pedidos (600 com erro), amostra de 10:")
print(" k   hipergeométrica   binomial")
for k in range(0, 5):
    print(f" {k}      {hiper(k, 5000, 600, 10):.4f}         {binom(k, 10, 0.12):.4f}")

print()
print("P(pelo menos 1 erro) em uma amostra de 10:")
print(f"  base de 5.000 — hipergeométrica: {1 - hiper(0, 5000, 600, 10):.4f}   binomial: {1 - binom(0, 10, 0.12):.4f}")
print(f"  base de    50 — hipergeométrica: {1 - hiper(0, 50, 6, 10):.4f}   binomial: {1 - binom(0, 10, 0.12):.4f}")

**Como ler:**
- Na base de **5.000**, as duas colunas coincidem até a terceira casa decimal (por exemplo, 0,2782 contra 0,2785 para $k = 0$). A binomial é uma aproximação excelente.
- Na base de **50**, a diferença aparece: 0,7585 pela hipergeométrica contra 0,7215 pela binomial. Com uma população pequena, cada pedido retirado muda bastante a composição do que sobra, e a binomial erra.

**Regra prática:** quando a amostra é uma parte **pequena** da população (aqui, 10 de 5.000), dá para usar a binomial no lugar da hipergeométrica. É o que se faz o tempo todo em ciência de dados: amostras de bases com milhões de registros são tratadas como binomiais.

## Para fechar

Neste notebook, o time de dados usou as duas ferramentas da Aula 10 para resolver problemas reais de uma empresa:

- **Hipergeométrica:** sorteio **sem reposição** de uma população finita — a auditoria de pedidos, incluindo a decisão de **quantos** auditar para atingir uma meta;
- **Binomial:** $n$ ensaios **independentes** com a mesma $p$ — a campanha de e-mail;
- **A ponte entre as duas:** quando a população é muito maior que a amostra, a binomial aproxima bem a hipergeométrica.

Na **Aula 11** vem a **distribuição de Poisson**, que conta eventos num intervalo de tempo — por exemplo, quantos pedidos chegam à loja por minuto.